# Substation Shifting Proportionality Test

**Question:** Is the amount of load shifting per substation (Tempo − Control, TOU − Control) proportional to the number of households in that substation?

**Tests used:**
- **Pearson correlation** — tests linear association between n_households and total shifting
- **OLS regression with intercept** — verifies whether intercept ≈ 0 (necessary condition for proportionality)
- **OLS regression through origin** — directly models `shifting = k × n_households`; `k` is the per-household shift rate

**Why not ANOVA?** ANOVA compares group means across categories. Substation is not a treatment group — it's a continuous predictor. ANOVA can't express a proportional relationship.

**Why not chi-square?** Chi-square requires categorical data. Both our variables are continuous (kWh, household counts). Binning them would be arbitrary and lossy.

**Red day only:** Tempo tariffs are colour-coded (red/white/blue). The highest-signal day is the single red day (2014-07-03), where the Tempo price incentive is strongest. Both scenarios are filtered to this day only.

**TOU tariff window — hours 13–19:** Inspection of per-hour signed differences (TOU − control) on the red day shows:
- Hours 1–12: zero difference (outside scheme)
- Hours 13–19: load strictly decreases (0 % of households see an increase) → **tariff window**
- Hours 20–24: load strictly increases (reallocated demand shifted out of the tariff window)

TOU shifting = `sum(ctrl_h13-19) − sum(tou_h13-19)` per substation (always ≥ 0; measures load shed during the tariff window).

n = 21 substations (one observation per substation)

In [ ]:
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

BASE_DIR    = '/home/user/capstone_visuals'
ZIP_PATH    = os.path.join(BASE_DIR, 'dist_net.zip')
CONTROL_CSV = os.path.join(BASE_DIR, 'data', 'control_profile.csv')
TEMPO_CSV   = os.path.join(BASE_DIR, 'data', 'tempo_shifted_profile.csv')
TOU_ZIP     = os.path.join(BASE_DIR, 'data', 'tou_shifted_profile.zip')
EXTRACT_DIR = '/tmp/dist_net'
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')

RED_DAY       = '2014-07-03'
HOUR_COLS     = [str(i) for i in range(1, 25)]   # all 24 hours — Tempo
TOU_HOUR_COLS = [str(i) for i in range(13, 20)]  # TOU tariff window: hours 13-19

def normalize_hid(x):
    try:    return str(int(float(x)))
    except: return None

print('Imports OK')
print('Red day:      ', RED_DAY)
print('Tempo hours:  ', f'{HOUR_COLS[0]}-{HOUR_COLS[-1]}  ({len(HOUR_COLS)} hours)')
print('TOU window:   ', f'{TOU_HOUR_COLS[0]}-{TOU_HOUR_COLS[-1]}  ({len(TOU_HOUR_COLS)} hours)')

## 1. Build HID → Substation Mapping from Shapefiles

In [ ]:
# Extract network zip if needed
flag = os.path.join(EXTRACT_DIR, '.done')
if not os.path.exists(flag):
    print('Extracting dist_net.zip...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    open(flag, 'w').close()

# Load all H-nodes across substations to build HID → substation map
content = os.path.join(EXTRACT_DIR, 'content', 'output')
all_nodes = []

for folder in sorted(os.listdir(content)):
    fp = os.path.join(content, folder, f'{folder}-nodelist-HID.shp')
    if not os.path.exists(fp):
        continue
    gdf = gpd.read_file(fp)
    gdf['substation'] = folder
    all_nodes.append(gdf)

nodes = pd.concat(all_nodes, ignore_index=True)
h_nodes = nodes[nodes['label'] == 'H'].copy()
h_nodes['hid_key'] = h_nodes['hid'].apply(normalize_hid)
h_nodes = h_nodes.dropna(subset=['hid_key'])

# Household count per substation
hh_counts = h_nodes.groupby('substation')['hid_key'].nunique().rename('n_households')

print(f'Total H-nodes: {len(h_nodes):,}')
print(f'Substations:   {hh_counts.shape[0]}')
print()
print(hh_counts.sort_values(ascending=False).to_string())

## 2. Load CSVs and Compute Per-Substation Shifting (Red Day Only)

Both datasets are filtered to the single red day before any aggregation.

**Shifting metrics:**
- **Tempo (signed):** `sum(tempo_h1-24) − sum(ctrl_h1-24)` — net load change across the full day
- **Tempo (absolute):** `|signed shift|` — magnitude regardless of direction
- **TOU:** `sum(ctrl_h13-19) − sum(tou_h13-19)` — load shed during the tariff window (always ≥ 0 since TOU only suppresses load in those hours)

In [ ]:
# Load and immediately filter to red day
print('Loading control...')
ctrl = pd.read_csv(CONTROL_CSV)
ctrl['hid_key'] = ctrl['hid'].apply(normalize_hid)
ctrl = ctrl[ctrl['date'] == RED_DAY]

print('Loading tempo...')
tempo = pd.read_csv(TEMPO_CSV)
tempo['hid_key'] = tempo['hid'].apply(normalize_hid)
tempo = tempo[tempo['date'] == RED_DAY]

print('Loading TOU...')
with zipfile.ZipFile(TOU_ZIP) as z:
    with z.open('tou_shifted_profile.csv') as f:
        tou = pd.read_csv(f)
tou['hid_key'] = tou['hid'].apply(normalize_hid)
tou = tou[tou['date'] == RED_DAY]

print()
print(f'Control  — red day rows: {len(ctrl):,}   unique HIDs: {ctrl["hid_key"].nunique():,}')
print(f'Tempo    — red day rows: {len(tempo):,}   unique HIDs: {tempo["hid_key"].nunique():,}')
print(f'TOU      — red day rows: {len(tou):,}   unique HIDs: {tou["hid_key"].nunique():,}')

In [ ]:
# Merge HID → substation into each dataset
hid_sub = h_nodes[['hid_key', 'substation']].drop_duplicates(subset='hid_key')

def add_substation(df):
    return df.merge(hid_sub, on='hid_key', how='inner')

ctrl_s  = add_substation(ctrl)
tempo_s = add_substation(tempo)
tou_s   = add_substation(tou)

print(f'After merge — Control: {len(ctrl_s):,} rows | Tempo: {len(tempo_s):,} rows | TOU: {len(tou_s):,} rows')

In [ ]:
def substation_load(df, hour_cols):
    """Sum specified hours across all HID rows, grouped by substation."""
    df = df.copy()
    df['window_total'] = df[hour_cols].sum(axis=1)
    return df.groupby('substation')['window_total'].sum()

# Tempo: full 24h net shift
load_ctrl_full = substation_load(ctrl_s,  HOUR_COLS)
load_tempo     = substation_load(tempo_s, HOUR_COLS)

# TOU: load shed = ctrl - tou over tariff window hours 13-19
# (direction is always ctrl >= tou in those hours, so result is >= 0)
load_ctrl_tou  = substation_load(ctrl_s, TOU_HOUR_COLS)
load_tou       = substation_load(tou_s,  TOU_HOUR_COLS)

df = pd.DataFrame({
    'n_households':  hh_counts,
    'load_ctrl_full': load_ctrl_full,
    'load_tempo':     load_tempo,
    'load_ctrl_tou':  load_ctrl_tou,
    'load_tou_win':   load_tou,
}).dropna()

df['shift_tempo']     = df['load_tempo']    - df['load_ctrl_full']  # signed net
df['abs_shift_tempo'] = df['shift_tempo'].abs()
df['shift_tou']       = df['load_ctrl_tou'] - df['load_tou_win']    # load shed (>= 0)

print(f'Substations in analysis: {len(df)}')
print()
print(df[['n_households', 'shift_tempo', 'abs_shift_tempo', 'shift_tou']]
      .sort_values('n_households', ascending=False)
      .rename(columns={'shift_tou': 'tou_load_shed'})
      .to_string())

## 3. Pearson Correlation

Tests H₀: ρ = 0 (no linear association between n_households and shifting).  
Tempo: signed and absolute net shift (24h, red day). TOU: load shed in tariff window (h13–19, red day).

In [ ]:
def pearson_report(x, y, label_x, label_y):
    r, p = stats.pearsonr(x, y)
    n = len(x)
    z  = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    ci_lo, ci_hi = np.tanh(z - 1.96*se), np.tanh(z + 1.96*se)
    print('  %s ~ %s' % (label_y, label_x))
    print('    n = %d' % n)
    print('    r = %+.4f   (95%% CI: [%+.4f, %+.4f])' % (r, ci_lo, ci_hi))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '(ns)'
    print('    p = %.4f  %s' % (p, sig))
    print()
    return r, p

print('=== Pearson Correlation: n_households vs shifting (red day) ===')
print()
print('--- TEMPO ---')
pearson_report(df['n_households'], df['shift_tempo'],     'n_households', 'shift_tempo (signed, 24h)')
pearson_report(df['n_households'], df['abs_shift_tempo'], 'n_households', 'shift_tempo (absolute, 24h)')

print('--- TOU (load shed, tariff window h13-19) ---')
pearson_report(df['n_households'], df['shift_tou'], 'n_households', 'tou_load_shed')

## 4. OLS Regression

**With intercept:** checks whether the intercept is statistically distinguishable from zero.  
**Through origin (no intercept):** the strict proportionality model `shifting = k × n_households`.  

Uses `scipy.stats.linregress` (with intercept) and a manual OLS formula for the origin-constrained version.

In [ ]:
def ols_report(x, y, label_y, scenario):
    x_arr = x.values.astype(float)
    y_arr = y.values.astype(float)
    n = len(x_arr)

    res = stats.linregress(x_arr, y_arr)
    sig_slope = '***' if res.pvalue<0.001 else '**' if res.pvalue<0.01 else '*' if res.pvalue<0.05 else '(ns)'
    t_int = res.intercept / res.intercept_stderr
    print('  [%s] %s ~ n_households' % (scenario, label_y))
    print('  OLS WITH intercept:')
    print('    slope     = %+.6f  (se=%.6f,  p=%.4f %s)' % (res.slope, res.stderr, res.pvalue, sig_slope))
    print('    intercept = %+.4f  (se=%.4f,  t=%+.3f)' % (res.intercept, res.intercept_stderr, t_int))
    print('    R²        = %.4f' % (res.rvalue**2))

    k = np.dot(x_arr, y_arr) / np.dot(x_arr, x_arr)
    residuals = y_arr - k * x_arr
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_arr - y_arr.mean())**2)
    r2_origin = 1 - ss_res / ss_tot
    sigma2 = ss_res / (n - 1)
    se_k   = np.sqrt(sigma2 / np.dot(x_arr, x_arr))
    t_k    = k / se_k
    p_k    = 2 * stats.t.sf(abs(t_k), df=n-1)
    sig_k  = '***' if p_k<0.001 else '**' if p_k<0.01 else '*' if p_k<0.05 else '(ns)'
    print('  OLS THROUGH ORIGIN (strict proportionality):')
    print('    k (kWh/household) = %+.6f  (se=%.6f,  t=%+.3f,  p=%.4f %s)' % (k, se_k, t_k, p_k, sig_k))
    print('    R²                = %.4f' % r2_origin)
    print()
    return res, k, se_k

print('=== OLS Regression (red day) ===')
print()
print('--- TEMPO: SIGNED ---')
res_tempo,  k_tempo,  se_tempo  = ols_report(df['n_households'], df['shift_tempo'],     'shift_tempo',       'TEMPO')
print('--- TEMPO: ABSOLUTE ---')
res_atempo, k_atempo, se_atempo = ols_report(df['n_households'], df['abs_shift_tempo'], 'abs_shift_tempo',   'TEMPO')
print('--- TOU: LOAD SHED (h13-19) ---')
res_tou,    k_tou,    se_tou    = ols_report(df['n_households'], df['shift_tou'],        'tou_load_shed',     'TOU')

## 5. Visualisation

In [ ]:
BG   = '#06090f'
TXT  = '#c8d8e8'
DIM  = '#2a3f55'
C_TEMPO = '#ff8c00'
C_TOU   = '#00b4d8'
C_FIT   = '#ffffff'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'text.color': TXT, 'axes.labelcolor': TXT,
    'xtick.color': TXT, 'ytick.color': TXT,
    'axes.edgecolor': DIM, 'grid.color': DIM,
    'font.family': 'monospace', 'font.size': 9,
})

x_range = np.linspace(0, df['n_households'].max() * 1.05, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)
fig.suptitle('Substation Shifting vs. Household Count  |  Red Day (2014-07-03)\n'
             'Solid = OLS with intercept   |   Dashed = through-origin (proportionality)',
             color='white', fontsize=11, fontweight='bold', y=1.03)

configs = [
    (axes[0], df['shift_tempo'],     C_TEMPO, 'TEMPO',  'Signed Net Shift (kWh, 24h)',       res_tempo,  k_tempo ),
    (axes[1], df['abs_shift_tempo'], C_TEMPO, 'TEMPO',  'Absolute Net Shift (kWh, 24h)',     res_atempo, k_atempo),
    (axes[2], df['shift_tou'],       C_TOU,   'TOU',    'Load Shed (kWh, h13-19)',           res_tou,    k_tou   ),
]

for ax, y_data, color, scenario, ylabel, res_obj, k_val in configs:
    x_vals = df['n_households'].values
    y_vals = y_data.values

    r, p = stats.pearsonr(x_vals, y_vals)
    sig   = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    ax.scatter(x_vals, y_vals, color=color, s=55, zorder=5, alpha=0.85,
               edgecolors='white', linewidths=0.4)

    for sub, xi, yi in zip(df.index, x_vals, y_vals):
        ax.annotate(sub, (xi, yi), textcoords='offset points', xytext=(5, 3),
                    fontsize=5.5, color=TXT, alpha=0.7)

    ax.plot(x_range, res_obj.slope*x_range + res_obj.intercept,
            color=C_FIT, lw=1.3, alpha=0.7, zorder=4, label='OLS fit')
    ax.plot(x_range, k_val*x_range,
            color=C_FIT, lw=1.3, alpha=0.7, zorder=4, linestyle='--', label='Through origin')

    ax.axhline(0, color=DIM, lw=0.8, zorder=2)
    ax.set_xlabel('Households per substation', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title('%s  |  r = %+.3f  (p = %.3f, %s)' % (scenario, r, p, sig), fontsize=9)
    ax.grid(lw=0.35, alpha=0.5)
    ax.legend(fontsize=7, facecolor='#0c1622', edgecolor=DIM, labelcolor=TXT)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, '07_shifting_proportionality.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved to', out_path)

## 6. Summary Table

In [ ]:
rows = []
for scenario, measure, y_col, res_obj, k_val, se_k_val in [
    ('Tempo', 'Signed net (24h)',      'shift_tempo',     res_tempo,  k_tempo,  se_tempo ),
    ('Tempo', 'Absolute net (24h)',    'abs_shift_tempo', res_atempo, k_atempo, se_atempo),
    ('TOU',   'Load shed (h13-19)',    'shift_tou',       res_tou,    k_tou,    se_tou   ),
]:
    x = df['n_households'].values.astype(float)
    y = df[y_col].values.astype(float)
    r, p = stats.pearsonr(x, y)
    rows.append({
        'Scenario': scenario,
        'Shifting measure': measure,
        'Pearson r': round(r, 4),
        'p-value':  round(p, 4),
        'Sig. (a=0.05)': 'Yes' if p < 0.05 else 'No',
        'OLS slope': round(res_obj.slope, 6),
        'OLS intercept': round(res_obj.intercept, 4),
        'OLS R2': round(res_obj.rvalue**2, 4),
        'k (kWh/HH, origin)': round(k_val, 6),
        'k SE': round(se_k_val, 6),
    })

summary = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(summary.to_string(index=False))